In [3]:
import os
import pandas as pd
import numpy as np
import optuna
import torch
from sklearn.model_selection import train_test_split
from torch.optim import Adam
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss
import torchvision.transforms as transforms

from utilsClass import TargetFeature
from utils import process_data, set_random_seed
from utilsDataset import img_Dataset
from utilsNN import CNN_pretrain, ResNet18_layer4
from utilsTrain import train_model

set_random_seed()

# load dataset
poi_data = pd.read_csv("poi_dataset.csv")

# process + new data
poi_data_processed = process_data(poi_data)

# split data into train and test datasets
df_train, df_test = train_test_split(poi_data_processed, test_size = 0.2, random_state = 42)
print(f'Number of samples.')
print(f'Train dataset: {df_train.shape[0]}')
print(f'Test dataset: {df_test.shape[0]}')

preproc_target = TargetFeature(col1_name='Visits', col2_name='Likes_Dislikes')

X_train = np.array(df_train['main_image_path'])
y_train = preproc_target.fit_transform(df_train)
X_test = np.array(df_test['main_image_path'])
y_test = preproc_target.transform(df_test)

# split train data into train and validation datasets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.2, random_state = 42)

transform_ResNet18 = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])

train_dataset = img_Dataset(y_train, X_train, transform_img = transform_ResNet18)
val_dataset = img_Dataset(y_val, X_val, transform_img = transform_ResNet18)

def objective(trial):

    # hyperparameters to optimize
    learning_rate = trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True) # To optimize
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256]) # To optimize

    # Neural network configuration
    num_epochs = 10 
    criterion = CrossEntropyLoss() 
    model = CNN_pretrain(ResNet18_layer4) # Optimized parameter
    optimizer = Adam(model.parameters(), lr=learning_rate) # Optimized parameter
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # Optimized parameter
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False) # Optimized parameter
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # train model
    train_results = train_model(model, criterion, optimizer, num_epochs,
                                train_loader, val_loader, device, verbose = False)

    # save metrics
    trial.set_user_attr('train_results', train_results)

    return train_results['val_accs'][-1]

study = optuna.create_study(
    direction='maximize',
    storage='sqlite:///optuna_DL_exercise.db',  # Persistent storage
    study_name='resnet18_pretrained_features1',
    sampler=optuna.samplers.TPESampler(
        n_startup_trials = 10,
        n_ei_candidates = 24,
        seed=42),
    load_if_exists=True  # Continue if study exists
)

study.set_user_attr('script', 'resnet18_pretrained_features1.py')
study.set_user_attr('dataset', 'poi_dataset.csv')
study.set_user_attr('model_architecture', 'ResNet18 pretrained optimization for layer4')
study.set_user_attr('split_dataset', '80(80train/20val)20test/seed42')
study.set_user_attr('description', 'ResNet18 with pretrained weigths with optimization for layer4. Last layer changed to two levels classification')
study.set_user_attr('score', 'accuracy')
study.set_user_attr('target', 'mean value for MinMaxScaler(Visits) and MinMaxScaler(Likes_Dislikes)')
study.set_user_attr('img_transformation', 'ResNet transformation')
study.set_user_attr('numepochs', 10)
study.set_user_attr('criterion', 'CrossEntropyLoss')
study.set_user_attr('optimizer', 'Adam')

# Run trials
study.optimize(objective, n_trials=10)
print(f"Completed {len(study.trials)} trials")
print(f"Best score: {study.best_value:.4f}")
print(f"Best params: {study.best_trial.params}")


Number of samples.
Train dataset: 1255
Test dataset: 314


[I 2025-09-23 09:37:43,644] A new study created in RDB with name: resnet18_pretrained_features1
[I 2025-09-23 09:44:55,032] Trial 0 finished with value: 57.76892430278885 and parameters: {'dropout_rate': 0.18727005942368125, 'learning_rate': 0.07969454818643935, 'batch_size': 32}. Best is trial 0 with value: 57.76892430278885.
[I 2025-09-23 09:52:10,135] Trial 1 finished with value: 59.36254980079681 and parameters: {'dropout_rate': 0.02904180608409973, 'learning_rate': 0.05399484409787434, 'batch_size': 256}. Best is trial 1 with value: 59.36254980079681.
[I 2025-09-23 09:59:13,219] Trial 2 finished with value: 60.55776892430279 and parameters: {'dropout_rate': 0.41622132040021087, 'learning_rate': 0.0026587543983272706, 'batch_size': 256}. Best is trial 2 with value: 60.55776892430279.
[I 2025-09-23 10:06:36,849] Trial 3 finished with value: 59.7609561752988 and parameters: {'dropout_rate': 0.21597250932105788, 'learning_rate': 0.0038234752246751854, 'batch_size': 32}. Best is trial 

KeyboardInterrupt: 

In [1]:
from utilsNN import ResNet18_layer4

model = ResNet18_layer4()

In [2]:
# Check which layers are frozen/trainable
for name, param in model.named_parameters():
    print(f"{name}: requires_grad = {param.requires_grad}")

conv1.weight: requires_grad = False
bn1.weight: requires_grad = False
bn1.bias: requires_grad = False
layer1.0.conv1.weight: requires_grad = False
layer1.0.bn1.weight: requires_grad = False
layer1.0.bn1.bias: requires_grad = False
layer1.0.conv2.weight: requires_grad = False
layer1.0.bn2.weight: requires_grad = False
layer1.0.bn2.bias: requires_grad = False
layer1.1.conv1.weight: requires_grad = False
layer1.1.bn1.weight: requires_grad = False
layer1.1.bn1.bias: requires_grad = False
layer1.1.conv2.weight: requires_grad = False
layer1.1.bn2.weight: requires_grad = False
layer1.1.bn2.bias: requires_grad = False
layer2.0.conv1.weight: requires_grad = False
layer2.0.bn1.weight: requires_grad = False
layer2.0.bn1.bias: requires_grad = False
layer2.0.conv2.weight: requires_grad = False
layer2.0.bn2.weight: requires_grad = False
layer2.0.bn2.bias: requires_grad = False
layer2.0.downsample.0.weight: requires_grad = False
layer2.0.downsample.1.weight: requires_grad = False
layer2.0.downsample.